# Credit EDA and Analysis


## Exploração e análise de dados de crédito com SQL
Esse notebook faz parte do curso SQL para análise de dados da EBAC.

A tabela foi criada no AWS Athena junto com o S3 Bucket com uma versão dos dados disponibilizados em: <https://github.com/andre-marcos-perez/ebac-course-utils/tree/main/dataset>, versão utilizada no projeto teve o total de linhas de 10.128 linhas reduzidas para 2.564.

### Dados:


* idade = idade do clientesexo = sexo do cliente (F ou M)
* dependentes = número de dependentes do cliente
* escolaridade = nível de escolaridade do clientes
* salario_anual = faixa salarial do cliente
* tipo_cartao = tipo de cartao do cliente
* qtd_produtos = quantidade de produtos comprados nos últimos 12 meses
* iteracoes_12m = quantidade de iterações/transacoes nos ultimos 12 meses
* meses_inativo_12m = quantidade de meses que o cliente ficou inativo
* limite_credito = limite de credito do cliente
* valor_transacoes_12m = valor das transações dos ultimos 12 meses
* qtd_transacoes_12m = quantidade de transacoes dos ultimos 12 meses


**Criação tabela no AWS Athena**

Query: CREATE EXTERNAL TABLE IF NOT EXISTS default.credito ( 
  `idade` int,
  `sexo` string,
  `dependentes` int,
  `escolaridade` string,
  `estado_civil` string,
  `salario_anual` string,
  `tipo_cartao` string, 
  `qtd_produtos` bigint,
  `iteracoes_12m` int,
  `meses_inativo_12m` int,
  `limite_credito` float,
  `valor_transacoes_12m` float,
  `qtd_transacoes_12m` int 
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe'
WITH SERDEPROPERTIES (
  'serialization.format' = ',',
  'field.delim' = ','
) LOCATION <meu local no S3 de bucket criado para projeto>
TBLPROPERTIES ('has_encrypted_data'='false');


## Exploração de dados:

**Qual a quantidade de informações temos na nossa base de dados?**

Query: SELECT count(*) FROM credito

Resposta: 2564 linhas

**Visualização dos dados:**

![Imagem preview table](https://raw.githubusercontent.com/nathalia-de-sa/Credit-EDA-and-Analysis/refs/heads/main/01%20-%20Preview%20table.png)

Nessa primeira visualização onde vemos 10 linhas, podemos observar que existem dados nulos "na" na coluna escolaridade e estado civil.
É importante analisar os dados de todas colunas para não inferir em uma analise distorcida por valores nulos.

**Feature escolaridade**

![escolaridade](https://raw.githubusercontent.com/nathalia-de-sa/Credit-EDA-and-Analysis/refs/heads/main/02%20-%20escolaridade.png)

**Feature estado_civil**

![estado civil](https://raw.githubusercontent.com/nathalia-de-sa/Credit-EDA-and-Analysis/refs/heads/main/03%20-%20estado_civil.png)

**Feature Salário**

![salario](https://raw.githubusercontent.com/nathalia-de-sa/Credit-EDA-and-Analysis/refs/heads/main/04%20-%20salario.png)

**Feature Tipo Cartão**

![cartao](https://raw.githubusercontent.com/nathalia-de-sa/Credit-EDA-and-Analysis/refs/heads/main/05%20-%20tipo%20cartao.png)

Insight: Com isso é possivel observar que temos dados nulos em todas as features, exceto no tipo_cartao, e no caso desse projeto não iremos utilizar média ou moda para o preenchimento desses dados, iremos expurgar os valores nulos no momento da análise.

## Análise de dados

Após entender o dataset e explorar quais informações possui, iremos agorar extrair informações e analisar os dados.

**Quantos clientes temos de cada faixa salarial?**

Query: select count(*) as qtd_faixa, salario_anual
from credito
group by salario_anual;


![qtd faixa salaria](https://raw.githubusercontent.com/nathalia-de-sa/Credit-EDA-and-Analysis/refs/heads/main/06%20-%20qtd_faixa%20salaria.png)

O maior volume de clientes está na menor faixa <$40K com isso podemos começar a entender o perfil de clientes desse dataset. 
E o impacto de clientes com dados nulo é de 10%

**Quantos clientes são homens e quantos são mulheres?**

Query: select count(*) as qtd_sexo, sexo
from credito
group by sexo;

![qtd_m_f](https://raw.githubusercontent.com/nathalia-de-sa/Credit-EDA-and-Analysis/refs/heads/main/08-%20qtd_sexo.png)

![grafico_m_f](https://raw.githubusercontent.com/nathalia-de-sa/Credit-EDA-and-Analysis/refs/heads/main/14%20-%20grafico_m_f.png)

61% dos clientes são homens

**Queremos focar o nosso marketing de maneira adequada para nossos clientes, qual será a idade deles?**

Query: select round(avg(idade), 1) as media_idade, min(idade) as min_idade, max(idade) as max_idade, sexo
from credito
group by sexo;


![media idade](https://raw.githubusercontent.com/nathalia-de-sa/Credit-EDA-and-Analysis/refs/heads/main/07%20-%20media_idade.png)

A média de idade dos clientes homens e mulheres são bem próximas de 46 anos. 
Com isso, essa análise não tem nenhum impacto no direcionamento etário entre homens e mulheres, ofertando produtos para a faixa dos 46 anos atenderá bem os dois públicos.

**Qual o valor máximo e mínimo de transação?**

Query: select round(min(valor_transacoes_12m), 2) as transacao_minima, max(valor_transacoes_12m) as transacao_minima
from credito;


![min_max_transacao](https://raw.githubusercontent.com/nathalia-de-sa/Credit-EDA-and-Analysis/refs/heads/main/09%20-%20max_min_transacao.png)

Alta amplitude de transação, valor minimo e maximo mostra variedade nos gastos.

**Quais as características dos clientes que possuem os maiores creditos?**

Query: select round(max(limite_credito), 2) as limite_credito, escolaridade, tipo_cartao, sexo
from credito
where escolaridade != 'na' and tipo_cartao != 'na' group by escolaridade, tipo_cartao, sexo 
order by limite_credito 
desc limit 10


![maiores_transacoes](https://raw.githubusercontent.com/nathalia-de-sa/Credit-EDA-and-Analysis/refs/heads/main/10%20-%20caract_max_credito.png)

Entre os top 10 clientes com limite de crédito podemos observar a predominancia de clientes do sexo masculino. Em relação a escolaridade não tem um impacto direto no limite, visto que temos variedade de escolaridade no ranking.

**Quais as características dos clientes que possuem os menores creditos?**

Query: select round(max(limite_credito), 2) as limite_credito, escolaridade, tipo_cartao, sexo
from credito
where escolaridade != 'na' and tipo_cartao != 'na' group by escolaridade, tipo_cartao, sexo 
order by limite_credito 
asc limit 10

![menores_transacoes](https://raw.githubusercontent.com/nathalia-de-sa/Credit-EDA-and-Analysis/refs/heads/main/11%20-%20caract_min_credito.png)

É possivel notar que os menores valores de créditos estão com clientes do sexo feminino. E também neste 10 clientes com menores limites de crédito nenhum possui cartão platinum

**Clientes de qual sexo gastam mais?**

Query: select round(max(valor_transacoes_12m), 2) as maior_valor_gasto, round(avg(valor_transacoes_12m), 2) as media_valor_gasto, round(min(valor_transacoes_12m), 2) as min_valor_gasto, sexo 
from credito 
group by sexo


![gastam_m_f](https://raw.githubusercontent.com/nathalia-de-sa/Credit-EDA-and-Analysis/refs/heads/main/12%20-%20m_f_max_min.png)

O sexo dos clientes não interferem no comportamento de gastos nesse dataset

**O salário impacta no limite?**

Query: select round(avg(qtd_produtos), 1) as qts_produtos, round(avg(valor_transacoes_12m), 2) as media_valor_transacoes, round(avg(limite_credito), 2) as media_limite, sexo, salario_anual 
from credito 
where salario_anual != 'na' 
group by sexo, salario_anual 
order by avg(valor_transacoes_12m) desc;


![impato_salario](https://raw.githubusercontent.com/nathalia-de-sa/Credit-EDA-and-Analysis/refs/heads/main/13%20-%20impacto_poder_compra.png)

Sim, clientes com menor faixa salarial possuem limites menores mesmo tendo media de valor de transações semelhantes

## Conclusão

Essas foram algumas análises extraídas do dataset de crédito. Abaixo estão os principais insights identificados:

* **Perfil dos clientes:**

A maioria dos clientes possui renda anual de até 40K.

O público é predominantemente masculino.

* **Escolaridade:**

A escolaridade não aparenta influenciar significativamente no limite de crédito ou tipo de cartão

* **Limite de crédito por perfil:**

Clientes com maiores limites são, em sua maioria, homens.

Clientes com menores limites são, em sua maioria, mulheres.

Entre os clientes com menores limites, não há presença de cartão Platinum.

* **Renda e limite de crédito:**

A faixa salarial tem forte impacto no limite de crédito concedido.

Não existem mulheres com salário anual acima de 60K na base analisada.
